# 04 — Condition Label Preparation

**Project:** Adaptive Deep Learning-Based Road Vision Enhancement and Risk-Aware Object Detection

This notebook prepares the condition labels required by the later Adaptive Road Vision pipeline.

### Input
`datasets/processed/metadata/bdd100k_processed_metadata.csv`

### Outputs
- `datasets/processed/metadata/condition_labels.csv`
- `datasets/processed/metadata/condition_labels.json`
- `datasets/processed/metadata/cyclegan_source_images.csv`

### Conditions
- Day
- Night
- Rain
- Fog
- Clear
- Unknown

### Important
This notebook does **not** create synthetic fog and does **not** modify the original BDD100K images.

Run Notebook 03 successfully first. It should produce the metadata CSV with the image/annotation matching completed.


In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

# ============================================================
# PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path(r"C:\Users\ankit\Desktop\Adaptive_Road_Vision")

DATASETS = PROJECT_ROOT / "datasets"
PROCESSED = DATASETS / "processed"
METADATA_DIR = PROCESSED / "metadata"

INPUT_CSV = METADATA_DIR / "bdd100k_processed_metadata.csv"
INPUT_JSON = METADATA_DIR / "bdd100k_processed_metadata.json"

OUTPUT_CSV = METADATA_DIR / "condition_labels.csv"
OUTPUT_JSON = METADATA_DIR / "condition_labels.json"
CYCLEGAN_CSV = METADATA_DIR / "cyclegan_source_images.csv"

METADATA_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT ROOT :", PROJECT_ROOT)
print("METADATA DIR :", METADATA_DIR)
print("INPUT CSV    :", INPUT_CSV)
print("INPUT EXISTS :", INPUT_CSV.exists())


PROJECT ROOT : C:\Users\ankit\Desktop\Adaptive_Road_Vision
METADATA DIR : C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\processed\metadata
INPUT CSV    : C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\processed\metadata\bdd100k_processed_metadata.csv
INPUT EXISTS : True


## 1. Check the required input file

Notebook 04 depends on the CSV generated by Notebook 03.

If `INPUT EXISTS` is `False`, stop here and fix Notebook 03 first.


In [2]:
from pathlib import Path

# ============================================================
# NOTEBOOK 04 — CHECK INPUT METADATA FILE
# ============================================================

# Project paths
PROJECT_ROOT = Path(r"C:\Users\ankit\Desktop\Adaptive_Road_Vision")

METADATA_DIR = (
    PROJECT_ROOT
    / "datasets"
    / "processed"
    / "metadata"
)

INPUT_CSV = METADATA_DIR / "bdd100k_processed_metadata.csv"

print("=" * 70)
print("METADATA INPUT CHECK")
print("=" * 70)

print(f"PROJECT ROOT : {PROJECT_ROOT}")
print(f"METADATA DIR : {METADATA_DIR}")
print(f"INPUT CSV    : {INPUT_CSV}")
print(f"INPUT EXISTS : {INPUT_CSV.exists()}")

# ------------------------------------------------------------
# Check project directory
# ------------------------------------------------------------

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"""
PROJECT ROOT DOES NOT EXIST.

Expected:
{PROJECT_ROOT}

Please check that the Adaptive_Road_Vision folder exists
at the above location.
"""
    )

# ------------------------------------------------------------
# Create metadata directory if it does not exist
# ------------------------------------------------------------

METADATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nMetadata directory ready:")
print(METADATA_DIR)

# ------------------------------------------------------------
# Check whether required CSV exists
# ------------------------------------------------------------

if INPUT_CSV.exists():

    print("\n" + "=" * 70)
    print("SUCCESS — INPUT METADATA FILE FOUND")
    print("=" * 70)

    print(f"\nFile:")
    print(INPUT_CSV)

else:

    print("\n" + "=" * 70)
    print("ERROR — INPUT METADATA FILE NOT FOUND")
    print("=" * 70)

    # Search for possible CSV files in metadata folder
    csv_files = list(METADATA_DIR.glob("*.csv"))

    if csv_files:

        print("\nCSV files currently found in metadata folder:")

        for file in csv_files:
            print(f"  - {file.name}")

        print("\nThe expected file is:")
        print("  bdd100k_processed_metadata.csv")

        raise FileNotFoundError(
            f"""
The required metadata file was not found:

{INPUT_CSV}

However, other CSV files were found in the metadata folder.

Please check Notebook 03_jsonl_annotation_processing.ipynb
and make sure it successfully creates:

bdd100k_processed_metadata.csv
"""
        )

    else:

        raise FileNotFoundError(
            f"""
NO CSV FILES WERE FOUND IN THE METADATA FOLDER.

Expected folder:
{METADATA_DIR}

Expected file:
{INPUT_CSV}

This means Notebook 03_jsonl_annotation_processing.ipynb
has not successfully generated the metadata CSV yet.

IMPORTANT:
Run Notebook 03 completely first.

Notebook 03 must finish with:

Indexed images     : 100,000
Annotation records : 100,000
Matched records    : 100,000
Unmatched records  : 0

Then it should create:

{INPUT_CSV}

After that, run this Notebook 04 cell again.
"""
        )

METADATA INPUT CHECK
PROJECT ROOT : C:\Users\ankit\Desktop\Adaptive_Road_Vision
METADATA DIR : C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\processed\metadata
INPUT CSV    : C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\processed\metadata\bdd100k_processed_metadata.csv
INPUT EXISTS : True

Metadata directory ready:
C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\processed\metadata

SUCCESS — INPUT METADATA FILE FOUND

File:
C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\processed\metadata\bdd100k_processed_metadata.csv


In [3]:
# Load metadata

df = pd.read_csv(INPUT_CSV)

print("Metadata loaded successfully.")
print("Rows   :", f"{len(df):,}")
print("Columns:", len(df.columns))

print("\nColumn names:")
for column in df.columns:
    print(" -", column)

print("\nFirst 5 rows:")
display(df.head())


Metadata loaded successfully.
Rows   : 100,000
Columns: 12

Column names:
 - split
 - image_name
 - weather_raw
 - weather
 - time_raw
 - time_of_day
 - object_count
 - object_classes
 - annotation_file
 - project_condition
 - image_path
 - image_id

First 5 rows:
   split         image_name weather_raw weather   time_raw time_of_day  object_count                                                                                           object_classes                      annotation_file project_condition                                                                                                image_path           image_id
0  train  0000f77c-6257be58       clear   Clear    daytime         Day            12                            0000f77c-6257be58|area/drivable|car|lane/road curb|traffic light|traffic sign  labels\train\0000f77c-6257be58.json               Day  C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\raw\BDD100K\images\100k\train\0000f77c-6257be58.jpg  0000f77c-6257

## 2. Detect metadata columns

Notebook 03 may use slightly different names depending on the version used. This cell detects the required columns safely.


In [4]:
def detect_column(frame, candidates, required=True):
    normalized = {
        str(column).strip().lower(): column
        for column in frame.columns
    }

    for candidate in candidates:
        if candidate.lower() in normalized:
            return normalized[candidate.lower()]

    if required:
        raise ValueError(
            f"Could not find a required column. "
            f"Expected one of: {candidates}"
        )

    return None


split_col = detect_column(
    df,
    ["split", "dataset_split", "set"]
)

image_col = detect_column(
    df,
    ["image_name", "image_id", "filename", "file_name"]
)

weather_col = detect_column(
    df,
    ["weather", "weather_condition", "weather_label"],
    required=False
)

time_col = detect_column(
    df,
    ["time_of_day", "timeofday", "time"],
    required=False
)

annotation_col = detect_column(
    df,
    ["annotation_file", "label_file", "json_file"],
    required=False
)

print("Detected columns")
print("=" * 60)
print("Split      :", split_col)
print("Image      :", image_col)
print("Weather    :", weather_col)
print("Time       :", time_col)
print("Annotation :", annotation_col)


Detected columns
Split      : split
Image      : image_name
Weather    : weather
Time       : time_of_day
Annotation : annotation_file


## 3. Clean the raw metadata

The original values are preserved in `weather_raw` and `time_raw`. Normalized labels are created separately.


In [5]:
def clean_value(value):
    if pd.isna(value):
        return "Unknown"

    value = str(value).strip()

    if value == "":
        return "Unknown"

    return value


df[split_col] = (
    df[split_col]
    .map(clean_value)
    .str.lower()
)

df[image_col] = (
    df[image_col]
    .map(clean_value)
)

# If Notebook 03 already created normalized weather/time columns,
# use them. Otherwise create them from the raw columns.

if weather_col is None:
    df["weather_raw"] = "Unknown"
    weather_col = "weather_raw"
else:
    df[weather_col] = df[weather_col].map(clean_value)

if time_col is None:
    df["time_raw"] = "Unknown"
    time_col = "time_raw"
else:
    df[time_col] = df[time_col].map(clean_value)

print("Metadata cleaned.")


Metadata cleaned.


## 4. Normalize weather labels

The normalization keeps Fog and Rain explicitly available for the adaptive enhancement branches.


In [6]:
def normalize_weather(value):
    value = str(value).strip().lower()

    if value in {
        "fog",
        "foggy",
        "mist",
        "haze"
    }:
        return "Fog"

    if value in {
        "rain",
        "rainy",
        "drizzle",
        "storm",
        "wet"
    }:
        return "Rain"

    if value in {
        "clear",
        "sunny",
        "cloudy",
        "overcast"
    }:
        return "Clear"

    if value in {
        "snow",
        "snowy"
    }:
        return "Snow"

    return "Unknown"


df["weather_label"] = (
    df[weather_col]
    .map(normalize_weather)
)

print("Weather labels:")
display(df["weather_label"].value_counts(dropna=False))


Weather labels:
weather_label
Clear      73170
Unknown    11631
Snow        7891
Rain        7127
Fog          181


## 5. Normalize Day/Night labels


In [7]:
def normalize_time(value):
    value = str(value).strip().lower()

    if value in {
        "day",
        "daytime"
    }:
        return "Day"

    if value in {
        "night",
        "nighttime"
    }:
        return "Night"

    return "Unknown"


df["time_label"] = (
    df[time_col]
    .map(normalize_time)
)

print("Time labels:")
display(df["time_label"].value_counts(dropna=False))


Time labels:
time_label
Day        52504
Night      47280
Unknown      216


## 6. Create the main condition label

Priority used by this project:

1. Fog
2. Rain
3. Night
4. Day
5. Clear
6. Unknown

Fog and Rain are prioritized because they require dedicated restoration/enhancement processing later.


In [8]:
def make_condition(row):
    weather = row["weather_label"]
    time = row["time_label"]

    if weather == "Fog":
        return "Fog"

    if weather == "Rain":
        return "Rain"

    if time == "Night":
        return "Night"

    if time == "Day":
        return "Day"

    if weather == "Clear":
        return "Clear"

    return "Unknown"


df["condition"] = df.apply(
    make_condition,
    axis=1
)

print("Main condition counts:")
display(df["condition"].value_counts(dropna=False))


Main condition counts:
condition
Day        48894
Night      43597
Rain        7127
Unknown      197
Fog          181
Clear          4


## 7. Verify the original 70K / 10K / 20K split

This notebook does not change the original split.

Expected:
- Train = 70,000
- Validation = 10,000
- Test = 20,000


In [9]:
split_counts = (
    df[split_col]
    .value_counts()
    .sort_index()
)

print("Current dataset split:")
display(split_counts)

expected_splits = {
    "train": 70000,
    "val": 10000,
    "test": 20000
}

print("\nVerification:")
for split_name, expected_count in expected_splits.items():

    actual_count = int(
        split_counts.get(split_name, 0)
    )

    if actual_count == expected_count:
        print(
            f"PASS  {split_name:5s}: "
            f"{actual_count:,}"
        )
    else:
        print(
            f"CHECK {split_name:5s}: "
            f"{actual_count:,} "
            f"(expected {expected_count:,})"
        )


Current dataset split:
split
test     20000
train    70000
val      10000

Verification:
PASS  train: 70,000
PASS  val  : 10,000
PASS  test : 20,000


## 8. Condition × Split analysis


In [10]:
condition_by_split = pd.crosstab(
    df[split_col],
    df["condition"]
)

print("Condition × Split:")
display(condition_by_split)

print("\nWeather × Time:")
weather_time = pd.crosstab(
    df["weather_label"],
    df["time_label"]
)
display(weather_time)


Condition × Split:
condition  Clear    Day  Fog  Night  Rain  Unknown
split                                             
test           1   9817   38   8798  1306       40
train          2  34220  130  30441  5083      124
val            1   4857   13   4358   738       33

Weather × Time:
time_label       Day  Night  Unknown
weather_label                       
Clear          34749  38417        4
Fog               68    113        0
Rain            3542   3570       15
Snow            4085   3803        3
Unknown        10060   1377      194


## 9. Inspect the important Fog / Rain / Night groups

These counts help verify that the metadata is suitable for the later adaptive enhancement and CycleGAN stages.


In [11]:
train_df = df[
    df[split_col] == "train"
].copy()

def count_group(weather, time):
    return int(
        (
            (train_df["weather_label"] == weather) &
            (train_df["time_label"] == time)
        ).sum()
    )


group_counts = {
    "Clear + Day": count_group("Clear", "Day"),
    "Clear + Night": count_group("Clear", "Night"),
    "Rain + Day": count_group("Rain", "Day"),
    "Rain + Night": count_group("Rain", "Night"),
    "Fog + Day": count_group("Fog", "Day"),
    "Fog + Night": count_group("Fog", "Night"),
}

group_table = pd.DataFrame.from_dict(
    group_counts,
    orient="index",
    columns=["training_images"]
)

display(group_table)


               training_images
Clear + Day              24306
Clear + Night            26774
Rain + Day                2532
Rain + Night              2538
Fog + Day                   48
Fog + Night                 82


## 10. Select CycleGAN source images

Planned synthetic fog augmentation:

- 1,500 **Clear + Day** training images
- 1,500 **Clear + Night** training images
- Total = 3,000 source images

Only the **training split** is eligible.

Validation and test images are never selected as CycleGAN source images.


In [12]:
SEED = 42

DAY_TARGET = 1500
NIGHT_TARGET = 1500

day_pool = train_df[
    (train_df["weather_label"] == "Clear") &
    (train_df["time_label"] == "Day")
].copy()

night_pool = train_df[
    (train_df["weather_label"] == "Clear") &
    (train_df["time_label"] == "Night")
].copy()

print("Clear + Day training pool  :", f"{len(day_pool):,}")
print("Clear + Night training pool:", f"{len(night_pool):,}")

if len(day_pool) < DAY_TARGET:
    raise ValueError(
        f"Not enough Clear+Day images. "
        f"Found {len(day_pool)}, need {DAY_TARGET}."
    )

if len(night_pool) < NIGHT_TARGET:
    raise ValueError(
        f"Not enough Clear+Night images. "
        f"Found {len(night_pool)}, need {NIGHT_TARGET}."
    )

print("\nBoth source pools are large enough.")


Clear + Day training pool  : 24,306
Clear + Night training pool: 26,774

Both source pools are large enough.


In [13]:
day_sources = day_pool.sample(
    n=DAY_TARGET,
    random_state=SEED
).copy()

night_sources = night_pool.sample(
    n=NIGHT_TARGET,
    random_state=SEED
).copy()

day_sources["cyclegan_source_domain"] = "day"
day_sources["synthetic_target_domain"] = "fog"

night_sources["cyclegan_source_domain"] = "night"
night_sources["synthetic_target_domain"] = "fog"

cyclegan_sources = pd.concat(
    [day_sources, night_sources],
    ignore_index=True
)

print("CycleGAN source selection:")
print(
    "Day   :",
    len(day_sources)
)
print(
    "Night :",
    len(night_sources)
)
print(
    "Total :",
    len(cyclegan_sources)
)


CycleGAN source selection:
Day   : 1500
Night : 1500
Total : 3000


## 11. Save CycleGAN source metadata


In [14]:
source_columns = [
    split_col,
    image_col,
    "weather_label",
    "time_label",
    "condition",
    "cyclegan_source_domain",
    "synthetic_target_domain"
]

if annotation_col is not None and annotation_col in cyclegan_sources.columns:
    source_columns.append(annotation_col)

cyclegan_sources_to_save = cyclegan_sources[
    source_columns
].copy()

cyclegan_sources_to_save = (
    cyclegan_sources_to_save
    .rename(
        columns={
            split_col: "split",
            image_col: "image_name"
        }
    )
)

cyclegan_sources_to_save.to_csv(
    CYCLEGAN_CSV,
    index=False,
    encoding="utf-8"
)

print("Saved CycleGAN source metadata:")
print(CYCLEGAN_CSV)

print(
    "\nFile exists:",
    CYCLEGAN_CSV.exists()
)


Saved CycleGAN source metadata:
C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\processed\metadata\cyclegan_source_images.csv

File exists: True


## 12. Create fixed numeric condition IDs

This mapping must remain unchanged in later condition-analysis training/evaluation notebooks.


In [15]:
CONDITION_TO_ID = {
    "Day": 0,
    "Night": 1,
    "Rain": 2,
    "Fog": 3,
    "Clear": 4,
    "Unknown": 5
}

df["condition_id"] = (
    df["condition"]
    .map(CONDITION_TO_ID)
)

if df["condition_id"].isna().any():

    unknown_conditions = sorted(
        df.loc[
            df["condition_id"].isna(),
            "condition"
        ].unique()
    )

    raise ValueError(
        f"Unmapped conditions found: "
        f"{unknown_conditions}"
    )

df["condition_id"] = (
    df["condition_id"]
    .astype(int)
)

print("Condition mapping:")
print(
    json.dumps(
        CONDITION_TO_ID,
        indent=4
    )
)


Condition mapping:
{
    "Day": 0,
    "Night": 1,
    "Rain": 2,
    "Fog": 3,
    "Clear": 4,
    "Unknown": 5
}


## 13. Prepare the final condition-label table


In [16]:
final_columns = [
    split_col,
    image_col,
    weather_col,
    time_col,
    "weather_label",
    "time_label",
    "condition",
    "condition_id"
]

if annotation_col is not None and annotation_col in df.columns:
    final_columns.append(annotation_col)

condition_labels = df[
    final_columns
].copy()

condition_labels = (
    condition_labels
    .rename(
        columns={
            split_col: "split",
            image_col: "image_name",
            weather_col: "weather_raw",
            time_col: "time_raw"
        }
    )
)

# Remove duplicate image records if any exist.
before = len(condition_labels)

condition_labels = (
    condition_labels
    .drop_duplicates(
        subset=["split", "image_name"],
        keep="first"
    )
    .reset_index(drop=True)
)

after = len(condition_labels)

print("Rows before duplicate removal:", f"{before:,}")
print("Rows after duplicate removal :", f"{after:,}")
print("Duplicates removed            :", f"{before - after:,}")


Rows before duplicate removal: 100,000
Rows after duplicate removal : 100,000
Duplicates removed            : 0


## 14. Save final condition labels


In [17]:
condition_labels.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8"
)

with open(
    OUTPUT_JSON,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        condition_labels.to_dict(
            orient="records"
        ),
        file,
        indent=2,
        ensure_ascii=False
    )

print("FINAL FILES")
print("=" * 60)

print("Condition CSV :")
print(OUTPUT_CSV)

print("\nCondition JSON:")
print(OUTPUT_JSON)

print("\nCSV exists :", OUTPUT_CSV.exists())
print("JSON exists:", OUTPUT_JSON.exists())


FINAL FILES
Condition CSV :
C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\processed\metadata\condition_labels.csv

Condition JSON:
C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\processed\metadata\condition_labels.json

CSV exists : True
JSON exists: True


## 15. Final validation

All checks below should pass before moving to Notebook 05.


In [18]:
print("=" * 70)
print("FINAL NOTEBOOK 04 VALIDATION")
print("=" * 70)

print(
    "\nTotal condition records:",
    f"{len(condition_labels):,}"
)

print(
    "Duplicate split + image:",
    condition_labels.duplicated(
        ["split", "image_name"]
    ).sum()
)

print(
    "Missing condition labels:",
    condition_labels["condition"].isna().sum()
)

print("\nSplit counts:")
display(
    condition_labels["split"]
    .value_counts()
    .sort_index()
)

print("\nCondition counts:")
display(
    condition_labels["condition"]
    .value_counts()
)

print("\nCondition × Split:")
display(
    pd.crosstab(
        condition_labels["split"],
        condition_labels["condition"]
    )
)

print("\nCycleGAN source counts:")
display(
    cyclegan_sources_to_save[
        "cyclegan_source_domain"
    ].value_counts()
)


FINAL NOTEBOOK 04 VALIDATION

Total condition records: 100,000
Duplicate split + image: 0
Missing condition labels: 0

Split counts:
split
test     20000
train    70000
val      10000

Condition counts:
condition
Day        48894
Night      43597
Rain        7127
Unknown      197
Fog          181
Clear          4

Condition × Split:
condition  Clear    Day  Fog  Night  Rain  Unknown
split                                             
test           1   9817   38   8798  1306       40
train          2  34220  130  30441  5083      124
val            1   4857   13   4358   738       33

CycleGAN source counts:
cyclegan_source_domain
day      1500
night    1500


In [19]:
# ============================================================
# FINAL ASSERTIONS
# ============================================================

assert OUTPUT_CSV.exists(), (
    "condition_labels.csv was not created."
)

assert OUTPUT_JSON.exists(), (
    "condition_labels.json was not created."
)

assert CYCLEGAN_CSV.exists(), (
    "cyclegan_source_images.csv was not created."
)

assert (
    condition_labels
    .duplicated(["split", "image_name"])
    .sum()
    == 0
), "Duplicate split + image records found."

assert (
    condition_labels["condition"]
    .isna()
    .sum()
    == 0
), "Missing condition labels found."

assert (
    len(cyclegan_sources_to_save) == 3000
), (
    "CycleGAN source count is not 3,000."
)

assert (
    (
        cyclegan_sources_to_save[
            "cyclegan_source_domain"
        ] == "day"
    ).sum()
    == 1500
), "CycleGAN Day source count is not 1,500."

assert (
    (
        cyclegan_sources_to_save[
            "cyclegan_source_domain"
        ] == "night"
    ).sum()
    == 1500
), "CycleGAN Night source count is not 1,500."

assert (
    set(
        cyclegan_sources_to_save["split"].unique()
    )
    == {"train"}
), "CycleGAN sources must come only from training."

print("=" * 70)
print("SUCCESS")
print("=" * 70)

print("Notebook 04 completed successfully.")
print()
print("Created:")
print("1.", OUTPUT_CSV)
print("2.", OUTPUT_JSON)
print("3.", CYCLEGAN_CSV)
print()
print("CycleGAN sources:")
print("- Clear + Day   : 1,500")
print("- Clear + Night : 1,500")
print("- Total         : 3,000")
print()
print("Next notebook: 05_train_val_test_split.ipynb")


SUCCESS
Notebook 04 completed successfully.

Created:
1. C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\processed\metadata\condition_labels.csv
2. C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\processed\metadata\condition_labels.json
3. C:\Users\ankit\Desktop\Adaptive_Road_Vision\datasets\processed\metadata\cyclegan_source_images.csv

CycleGAN sources:
- Clear + Day   : 1,500
- Clear + Night : 1,500
- Total         : 3,000

Next notebook: 05_train_val_test_split.ipynb


# Notebook 04 Complete

If the final cell prints **SUCCESS**, the condition-label preparation stage is complete.

### Files created

```text
datasets/
└── processed/
    └── metadata/
        ├── bdd100k_processed_metadata.csv
        ├── condition_labels.csv
        ├── condition_labels.json
        └── cyclegan_source_images.csv
```

### Next stage

Proceed to:

`05_train_val_test_split.ipynb`

Do not generate synthetic fog from validation or test images.
